In [1]:
import reframed
import pandas as pd
from pathlib import Path
import numpy as np
import sys

sys.path.append('../../code/7_GEM_reconstruction')

# from ng_utils import *
# from ng_tests import *
# from model_qc_and_polish import curate_model, remove_duplicated_metabolites, remove_duplicated_reactions, test_balance


from collections import defaultdict


In [2]:
repo_folder = Path("../..")
data_folder = repo_folder / "data" / "7_GEM_reconstruction"
gapfilled_folder = data_folder / 'models/gapfilled_FBA'
polished_folder = data_folder / 'models/polished'
# genome_folder = data_folder / 'genomes'

In [3]:
# universe = reframed.load_cbmodel(data_folder / 'carveme_universe_bacteria_fixed.xml')
# gp_folder = repo_folder / 'data' / '1_growth_phenotyping'
# selected_cs = pd.read_csv(gp_folder / 'selected_carbon_sources.csv', index_col=0)
# keep_exchanges = [f'R_EX_{cs}_e' for cs in selected_cs['BiGG ID'].tolist()]
# keep_exchanges += ['R_EX_o2_e', 'R_EX_h2o_e', 'R_EX_co2_e', 'R_EX_nh4_e', 'R_EX_pi_e', 'R_EX_so4_e', 'R_EX_k_e', 'R_EX_na1_e', 'R_EX_cl_e', 'R_EX_ca2_e', 'R_EX_mg2_e', 'R_EX_fe2_e', 'R_EX_fe3_e', 'R_EX_ni2_e']

In [4]:
model_abbrvs = ['At', 'Ct', 'Ml', 'Oa']

In [5]:
model_dict = {}
for abbrv in model_abbrvs:
    model_path = gapfilled_folder / f'{abbrv}_gapfilled.xml'
    model = reframed.load_cbmodel(str(model_path))
    model_dict[abbrv] = model

## Fix missing charges

In [6]:
charge_fn = data_folder / 'missing_metabolite_charges_bigg.csv'
charge_dict = pd.read_csv(charge_fn, index_col=0)['charge'].to_dict()

In [7]:
fixed_charges = {
    'ACP': -1,
    'fad': -2,
    'fadh2': -2,
    'ddcaACP': -1,
    'hd2coa': -4,
    'ddecoa': -4,
    'focytc': 0,
    'ode2coa': -4,
    # 'hdd7coa': 2,
    # '3oodecoa': 2,
    # '3hodecoa':2,
    'dde2coa': -4,
    'ddecoa': -5,
    # '3htdccoa': -4,
    '3hdd5coa': -4,
    'hdd4_2_coa': -4,
    "lnlc2coa": -4,
    'dec4coa':-4,
    'ptal': 0,
    'dann': 2,
    'dtbt':0,
    'btn': 0,
    'fdxo_2_2': 6,
    'fdxrd': 4,
    'fdxox': 5,
    'malACP': -2,
    'dc2coa': -4,
    'nonacoa': -2,
    'nona': 1,
    'poctacoa': -4,
    'php2coa': -4,
    'R_3hphpa': 1,
    'phxa2coa': -4,
    'ppt2coa': -6,
    '4atb2coa': -2,
    'decoa': -3,
    'R_3hnonaa': -1,
    'pta': -1,
    'pdcacoa': -4,
    'pocta2coa': -4,
    'R_3hpocta':-1,
    'tdecoa': -4,
    'R_3hdd6e': -1,
    'R_3hppta':-3,
    'R_3hphpa':-1,
    'R_3hphxa': -1,
    'ssaltpp': -3,
    'hp2coa': -4,
    'R_3hcmrs7e': -2,
    'R_3hhpa':-1,
    'lnlccoa':-2,
    'lnlc':1,
    'fe3dcit':-3,
    'seramp':0,
    'tag6p__D':-2,
    'mi3p__D':-2,
    'm2bcoa':0,
    '3h4atb': 1,
    'fmn':-2,
    'fmnh2':-2,
    'ribflv': 0,
    'hmbpp': -4,
    '23dhbzs2':-1,
    '23dhbzs3': -1,
    'fe3dhbzs3': 3,
    'trnaglu': 0,
    'cinmcoa': -4,
    # 'cinnm':-4
    # 'dmlz': -1,
    # 'db4p': -2,
    # '4r5au':-2,
    'salchs4': -1,
    'feenter': 2,
    'salc': -1,
    'dmso2': 4,
    'phenona': -3,
    'phehpa': -1,
    'abg4': -3,
    'val__D': -2,
    '3hpnonacoa': -4,
    '6ath2coa': -4,
    'hptcoa': -4,
}

equal = {
    'M_dde2coa_c': ['M_R_3hcddec5ecoa_c', 'M_3hddccoa_c', 'M_3oddccoa_c', 'M_decoa_c'],
    'M_3hpnonacoa_c': ['M_pnona2coa_c', 'M_R_3hpnonacoa_c', 'M_3opnonacoa_c', 'M_phpcoa_c'],
    'M_ddecoa_c': ['M_3otdccoa_c', 'M_3htdccoa_c', 'M_tde2coa_c', 'M_R_3hcmrs7ecoa_c'], 
    'M_3hdd5coa_c': ['M_tded5_2_coa_c', 'M_tded5coa_c', 'M_3ohdd7coa_c', 'M_3hhdd7coa_c', 'M_hdd7_2_coa_c'],
    # 'M_td2coa_c': ['M_tde_2Z_coa_c'],
    "M_lnlc2coa_c": ["M_3hlnlccoa_c", 'M_3olnlccoa_c', 'M_hd_7_10_coa_c', 'M_hd710_2_coa_c', 'M_3hhd710coa_c',
                     'M_3ohd710coa_c', 'M_td_5_8_coa_c', 'M_td58_2_coa_c', 'M_R_3htd58coa_c'],
    'M_dc2coa_c': ['M_R3hdec4coa_c', 'M_dec4_2_coa_c', 'M_3hdcoa_c'],
    'M_dec4coa_c': ['M_3odd6coa_c', 'M_3hdd6coa_c', 'M_dd6_2_coa_c', 'M_dd_3_6_coa_c', 'M_R_3hdd6coa_c',
                    'M_dd_3_6_coa_c', 'M_3ohd58coa_c', 'M_3hhd58coa_c'],
    'M_pocta2coa_c': ['M_R_3hpoctacoa_c', 'M_3hpoctacoa_c', 'M_3opoctacoa_c', 'M_phxacoa_c'],
    'M_php2coa_c': ['M_3hphpcoa_c', 'M_R_3hphpcoa_c','M_3ophpcoa_c', 'M_pptcoa_c'],
    # 'M_pro__D_c':['M_1p2cbxl_c'],
    'M_6ath2coa_c': ['M_R_3h6athcoa_c', 'M_3h6athcoa_c'],
    'M_hptcoa_c' : ['M_3ononacoa_c', 'M_3hnnacoa_c', 'M_nona2coa_c', 'M_R_3hnonacoa_c'],
    'M_hp2coa_c': ['M_3hhpcoa_c', 'M_R_3hhpcoa_c', 'M_3ohpcoa_c', 'M_ptcoa_c', 
                    'M_pt2coa_c', 'M_3hptcoa_c', 'M_3optcoa_c', 'M_ppcoa_c'],
    'M_4atb2coa_c': ['M_3h4atbcoa_c'],
    'M_hd2coa_c': ['M_3hhdccoa_c', 'M_3ohdccoa_c'],
    'M_ode2coa_c': ['M_3hodecoa_c', 'M_3oodecoa_c', 'M_hdd7coa_c'],
    'M_hdd4_2_coa_c': ['M_3hhdd4coa_c', 'M_3ohdd4coa_c', 'M_tde_2Z_coa_c'],
    
    'M_poctacoa_c': ['M_3opdecacoa_c', 'M_3hpdecacoa_c', 'M_pdca2coa_c', 'M_td2coa_c'],
    # 'M_decoa_c': ['M_3oddccoa_c'],
    'M_phxa2coa_c': ['M_R_3hphxacoa_c', 'M_3hphxacoa_c', 'M_3ophxacoa_c', 'M_pbcoa_c'],
    'M_ppt2coa_c': ['M_R_3hpptcoa_c', 'M_3hpptcoa_c', 'M_3opptcoa_c', 'M_phppcoa_c'],
}

for key, key_lst in equal.items():
    charge = fixed_charges.get(key[2:-2], None)
    if charge is not None:
        for short_id in key_lst:
            fixed_charges[short_id[2:-2]] = charge
            # charge_dict[short_id[2:-2]] = charge
            print(f'Setting fixed charge for {short_id[2:-2]} to {charge}')
    else:
        print(f'No fixed charge for {key} to set equal to {key_lst}')
charge_dict.update(fixed_charges)

Setting fixed charge for R_3hcddec5ecoa to -4
Setting fixed charge for 3hddccoa to -4
Setting fixed charge for 3oddccoa to -4
Setting fixed charge for decoa to -4
Setting fixed charge for pnona2coa to -4
Setting fixed charge for R_3hpnonacoa to -4
Setting fixed charge for 3opnonacoa to -4
Setting fixed charge for phpcoa to -4
Setting fixed charge for 3otdccoa to -5
Setting fixed charge for 3htdccoa to -5
Setting fixed charge for tde2coa to -5
Setting fixed charge for R_3hcmrs7ecoa to -5
Setting fixed charge for tded5_2_coa to -4
Setting fixed charge for tded5coa to -4
Setting fixed charge for 3ohdd7coa to -4
Setting fixed charge for 3hhdd7coa to -4
Setting fixed charge for hdd7_2_coa to -4
Setting fixed charge for 3hlnlccoa to -4
Setting fixed charge for 3olnlccoa to -4
Setting fixed charge for hd_7_10_coa to -4
Setting fixed charge for hd710_2_coa to -4
Setting fixed charge for 3hhd710coa to -4
Setting fixed charge for 3ohd710coa to -4
Setting fixed charge for td_5_8_coa to -4
Setting

In [8]:
STILL_MISSING = []

In [9]:
def fix_charge(model_dict):
    for abbrv, model in model_dict.items():
        for short_id, charge in fixed_charges.items():
            for end in ['_c', '_p', '_e']:
                met_id = f'M_{short_id}{end}'
                try:
                    met = model.metabolites[met_id]
                    # print(f'Setting fixed CHARGE for {met.id} in model {model.id} to {charge}')
                except KeyError:
                    continue
                else:
                    met.metadata['CHARGE'] = str(int(float(charge)))

        # Fill in missing charges from charge_dict
        for m_id in model.metabolites:
            met = model.metabolites[m_id]
            try:
                charge = int(float(met.metadata.get('CHARGE', np.nan)))
            except ValueError:
                charge = np.nan
            if np.isnan(charge):
                short_id = met.id[2:-2]
                new_charge = charge_dict.get(short_id, np.nan)
                if np.isfinite(new_charge):
                    # print(f'Setting CHARGE for {short_id} in model {model.id} to {new_charge}')
                    met.metadata['CHARGE'] = str(int(float(new_charge)))
                else:
                    print(f'No CHARGE for {short_id} in model {model.id} or in charge dict')
                    print(charge, new_charge)
                    # print(met.metadata)
                    STILL_MISSING.append(short_id)

        for m_id in model.metabolites:
            met = model.metabolites[m_id]
            try:
                met.metadata['CHARGE'] = str(int(float(met.metadata['CHARGE'])))
            except KeyError:
                print(f'No CHARGE for {met.id} in model {model.id} after fixing charges.')

# Small curations

In [10]:
def small_curations(model_dict):
    # At
    at = model_dict['At']
    at.remove_reactions(['R_AHSERL4', 'R_TSULR3e', 'R_MECDPDH4E', 'R_FEROpp'])
    # at.remove_reaction('R_BTS2')
    # at.reactions["R_TMDPK_1"].lb = -1000
    at.reactions['R_EX_glc__D_e'].lb = -10

    # Ct
    ct = model_dict['Ct']
    ct.reactions['R_EX_ac_e'].lb = -10
    ct.remove_reactions(['R_FEROpp'])
    # Ml
    ml = model_dict['Ml']
    ml.reactions['R_EX_glc__D_e'].lb = -10
    ml.reactions['R_EX_cys__L_e'].lb = -0.2
    ml.reactions['R_EX_btn_e'].lb = -0.001
    ml.reactions['R_EX_thm_e'].lb = -0.001
    # Oa
    oa = model_dict['Oa']
    oa.reactions['R_EX_thm_e'].lb = -0.001
    oa.reactions['R_EX_glc__D_e'].lb = -10


def remove_focytc_reactions(model_dict):
    # Redundant with focytC and ficytC
    for abbrv, model in model_dict.items():
        focytc_reactions = model.get_metabolite_consumers('M_focytc_c') + model.get_metabolite_producers('M_focytc_c')
        if focytc_reactions:
            print(f'Removing {len(focytc_reactions)} reactions involving M_focytc_c from model {abbrv}')
            model.remove_reactions(focytc_reactions)
            model.remove_metabolites(['M_focytc_c', 'M_ficytc_c'])
        else:
            print(f'No reactions involving M_focytC_c found in model {abbrv}')



# Add gene annotations

In [11]:
def add_ncbiprotein_annotations(model_dict):
    # Add NCBI protein IDs to gene annotations
    for abbrv, model in model_dict.items():
        i = 0
        for gene in model.genes.values():
            protein_id = gene.id[2:].replace('_', '.')
            if protein_id[:3] in ['WKL', 'WKT']:
                gene.metadata['ncbiprotein'] = protein_id
                i += 1
        print(f'Added {i} gene annotations to model {abbrv}')

    

# Add exchanges for metabolites known to be released by various microbes


In [12]:
likely_released_metabolites = list(set(sorted(['g6p', 'f6p','fdp','dhap','g3p','2pg','3pg','pep','r5p','ru5p__D','xu5p__D', 'e4p','cit','acon_C','icit','glu__L','ser__L',
                                               'asp__L','hom__L','tyr__L','gly','cys__L','pro__L','ala__L','met__L','val__L','arg__L','pyr','orot','fum','akg','ac','lys__L','gln__L','btn','thm'])))

In [13]:
def fix_metabolite_export(model_dict):
    for abbrv, model in model_dict.items():
        print(abbrv)
        constraints={f'R_EX_mal__L_e': (-10, 1000), 'R_EX_ac_e':(-10, 1000), 'R_EX_glc__D_e':(-10, 1000)}
        if abbrv == 'Ml':
            constraints['R_EX_cys__L_e'] = (-10, 1000)
            constraints['R_EX_btn_e'] = (-10, 1000)
            constraints['R_EX_thm_e'] = (-10, 1000)
        if abbrv == 'Oa':
            constraints['R_EX_thm_e'] = (-10, 1000)


        if abbrv in ['At', 'Ct']:
            model.add_reaction_from_str('R_THMP: M_h2o_c + M_thmmp_c --> M_pi_c + M_thm_c [0, 1000]')

        exchange_ids = {}
        for m_id in likely_released_metabolites:
            if abbrv == 'Oa' and m_id == 'thm':
                continue
            if abbrv == 'Ml' and m_id in ['btn', 'thm', 'cys__L']:
                continue 
            exchange_ids[m_id] = f'R_EX_{m_id}_e'

        for m_id, ex_id in exchange_ids.items():
            if not model.reactions.get(ex_id, None):
                model.add_reaction_from_str(f'{ex_id}: M_{m_id}_e <->  [0, 1000]')
        
        fva_res = reframed.FVA(model, reactions=exchange_ids.values(), constraints=constraints)

        reactions_to_add = []
        metabolites_to_fix = []
        
        print(fva_res)
        for r_id, (lb, ub) in fva_res.items():
            m_id = r_id[5:-2]
            if ub < 1e-6:
                print(f'{r_id} cant carry flux: [{lb}, {ub}]')
                metabolites_to_fix.append(m_id)
                pe_transport_str = f'R_t{m_id.upper()}pe: M_{m_id}_p <-> M_{m_id}_e [0,1000]'
                cp_transport_str = f'R_t{m_id.upper()}cp: M_{m_id}_c <-> M_{m_id}_p [0,1000]'

                # Add transport reaction from periplasm to extracellular
                modelc = model.copy()
                modelc.add_reaction_from_str(pe_transport_str)
                sol = reframed.FBA(modelc, objective = f'R_EX_{m_id}_e', constraints = constraints)
                if sol.fobj > 1e-6:
                    reactions_to_add.append(pe_transport_str)
                    # print(f'After adding transport reaction, {r_id} can carry flux: {sol.fobj}')
                    continue
                
                # Add transport reaction from cytoplasm to periplasm
                modelc = model.copy()
                modelc.add_reaction_from_str(cp_transport_str)
                sol = reframed.FBA(modelc, objective = f'R_EX_{m_id}_e', constraints = constraints)
                if sol.fobj > 1e-6:
                    reactions_to_add.append(cp_transport_str)
                    # print(f'After adding cytoplasm to periplasm transport reaction, {r_id} can carry flux: {sol.fobj}')
                    continue
                
                # Add both transport reactions
                modelc = model.copy()
                modelc.add_reaction_from_str(cp_transport_str)
                modelc.add_reaction_from_str(pe_transport_str)
                sol = reframed.FBA(modelc, objective = f'R_EX_{m_id}_e', constraints = constraints)
                if sol.fobj > 1e-6:
                    reactions_to_add.append(cp_transport_str)
                    reactions_to_add.append(pe_transport_str)
                    # print(f'After adding both transport reactions, {r_id} can carry flux: {sol.fobj}')
                    continue
                    
                print(f'No solution for {m_id}, {sol.fobj}')
        for r in reactions_to_add:
            print(f'Adding reaction: {r} to model {abbrv}')
            model.add_reaction_from_str(r)

        for m_id in metabolites_to_fix:
            met_id_c = f'M_{m_id}_c'
            met_id_p = f'M_{m_id}_p'
            met_id_e = f'M_{m_id}_e'
            met_c = model.metabolites[met_id_c]
            for met_id in [met_id_p, met_id_e]:
                met = model.metabolites[met_id]
                met.compartment = met_id[-1]
                met.metadata = met_c.metadata.copy()
    
            

In [14]:
def remove_sink_reactions(model_dict):
    for abbrv, model in model_dict.items():
        constraints={f'R_EX_mal__L_e': (-10, 1000), 'R_EX_ac_e':(-10, 1000), 'R_EX_glc__D_e':(-10, 1000)}
        if abbrv == 'Ml':
            constraints['R_EX_cys__L_e'] = (-10, 1000)
            constraints['R_EX_btn_e'] = (-10, 1000)
            constraints['R_EX_thm_e'] = (-10, 1000)
        if abbrv == 'Oa':
            constraints['R_EX_thm_e'] = (-10, 1000)
        sink_reactions = [r.id for r in model.reactions.values() if r.id.startswith('R_sink_')]
        wt_sol = reframed.FBA(model, constraints=constraints)
        for r_id in sink_reactions:
            new_dict = constraints.copy()
            new_dict.update({r_id: 0})
            sol = reframed.FBA(model, constraints=new_dict)
            if sol.fobj >= wt_sol.fobj - 1e-6:
                model.remove_reaction(r_id)
                print(f'Removed {r_id} sink reactions from model {model.id}')
            else:
                print(f'Keeping {r_id} sink reaction in model {model.id}')

In [15]:
def remove_disconnected_metabolites(model_dict):
    for abbrv, model in model_dict.items():
        disconnected_mets = [m.id for m in model.metabolites.values() if not model.get_metabolite_consumers(m.id) and not model.get_metabolite_producers(m.id)]
        if disconnected_mets:
            print(f'Removing {len(disconnected_mets)} disconnected metabolites from model {abbrv}')
            model.remove_metabolites(disconnected_mets)
        else:
            print(f'No disconnected metabolites found in model {abbrv}')

In [16]:
def remove_universally_blocked_reactions(model_dict):
    for abbrv, model in model_dict.items():
        print(f'Processing model {abbrv}')
        constraints = {}
        for r_id in model.get_exchange_reactions():
            constraints[r_id] = (-1000,1000)
        blocked_reactions = reframed.blocked_reactions(model, constraints=constraints)
        remove_reactions = [r for r in blocked_reactions if not r.startswith('R_EX_')]
        print(f'Number of blocked reactions: {len(blocked_reactions)}')
        print('Deleting blocked reactions:')
        print(remove_reactions)
        model.remove_reactions(remove_reactions)

In [17]:
# remove_disconnected_metabolites(model_dict)

# Save models

In [18]:
# small_curations(model_dict)
# for abbrv, model in model_dict.items():
#     polished_path = polished_folder / f'{abbrv}.xml'
#     reframed.save_cbmodel(model, str(polished_path))

In [19]:
m = model_dict['At'].metabolites['M_nad_c']

In [20]:
small_curations(model_dict)
add_ncbiprotein_annotations(model_dict)
fix_metabolite_export(model_dict)
fix_charge(model_dict)
remove_sink_reactions(model_dict)
remove_focytc_reactions(model_dict)
remove_universally_blocked_reactions(model_dict)
remove_disconnected_metabolites(model_dict)
for abbrv, model in model_dict.items():
    polished_path = polished_folder / f'{abbrv}.xml'
    reframed.save_cbmodel(model, str(polished_path))

Added 1411 gene annotations to model At
Added 1353 gene annotations to model Ct
Added 1021 gene annotations to model Ml
Added 1387 gene annotations to model Oa
At
Set parameter Username
Academic license - for non-commercial use only - expires 2026-11-28
{'R_EX_lys__L_e': [0.0, 14.423809523809531], 'R_EX_arg__L_e': [0.0, -0.0], 'R_EX_cys__L_e': [0.0, -0.0], 'R_EX_btn_e': [0.0, 2.363563829787234], 'R_EX_fdp_e': [0.0, -0.0], 'R_EX_f6p_e': [0.0, -0.0], 'R_EX_gln__L_e': [0.0, -0.0], 'R_EX_ser__L_e': [0.0, -0.0], 'R_EX_dhap_e': [0.0, -0.0], 'R_EX_val__L_e': [0.0, 17.847540983606564], 'R_EX_g6p_e': [0.0, -0.0], 'R_EX_orot_e': [0.0, -0.0], 'R_EX_asp__L_e': [0.0, -0.0], 'R_EX_icit_e': [0.0, -0.0], 'R_EX_pyr_e': [0.0, -0.0], 'R_EX_met__L_e': [0.0, -0.0], 'R_EX_r5p_e': [0.0, -0.0], 'R_EX_3pg_e': [0.0, -0.0], 'R_EX_akg_e': [0.0, -0.0], 'R_EX_ru5p__D_e': [0.0, -0.0], 'R_EX_acon_C_e': [0.0, -0.0], 'R_EX_pep_e': [0.0, -0.0], 'R_EX_tyr__L_e': [0.0, -0.0], 'R_EX_e4p_e': [0.0, -0.0], 'R_EX_pro__L_e': [0

# Code for testing purposes

In [21]:
at = model_dict['At']

In [22]:
at_df = test_balance(at)

NameError: name 'test_balance' is not defined

In [ ]:
len(at_df)

18

In [ ]:
j = 0
count_dict = defaultdict(int)
for i, row in at_df.iterrows():
    r_id = row['r_id']
    r = at.reactions[r_id]
    print(r, row['charge'])
    for met_id, coeff in r.stoichiometry.items():
        met = at.metabolites[met_id]
        count_dict[met_id] += 1
        print(f'  {met_id}: {coeff}, CHARGE={met.metadata.get("CHARGE", "NA")}')    
    j+=1
    # if j > 10:
    #     break

R_23CTI1: M_decoa_c --> M_dc2coa_c + M_h_c 1.0
  M_decoa_c: -1.0, CHARGE=-4
  M_dc2coa_c: 1.0, CHARGE=-4
  M_h_c: 1.0, CHARGE=1
R_ACOAD10f: M_fad_c + M_hptcoa_c + 2.0 M_h_c <-> M_fadh2_c + M_hp2coa_c -2.0
  M_fad_c: -1.0, CHARGE=-2
  M_hptcoa_c: -1.0, CHARGE=-4
  M_h_c: -2.0, CHARGE=1
  M_fadh2_c: 1.0, CHARGE=-2
  M_hp2coa_c: 1.0, CHARGE=-4
R_ACOAD12f: M_fad_c + M_pdcacoa_c + 2.0 M_h_c <-> M_fadh2_c + M_pdca2coa_c -2.0
  M_fad_c: -1.0, CHARGE=-2
  M_pdcacoa_c: -1.0, CHARGE=-4
  M_h_c: -2.0, CHARGE=1
  M_fadh2_c: 1.0, CHARGE=-2
  M_pdca2coa_c: 1.0, CHARGE=-4
R_ACOAD14f: M_fad_c + M_poctacoa_c + 2.0 M_h_c <-> M_fadh2_c + M_pocta2coa_c -2.0
  M_fad_c: -1.0, CHARGE=-2
  M_poctacoa_c: -1.0, CHARGE=-4
  M_h_c: -2.0, CHARGE=1
  M_fadh2_c: 1.0, CHARGE=-2
  M_pocta2coa_c: 1.0, CHARGE=-4
R_ACOAD15f: M_fad_c + M_phpcoa_c + 2.0 M_h_c <-> M_fadh2_c + M_php2coa_c -2.0
  M_fad_c: -1.0, CHARGE=-2
  M_phpcoa_c: -1.0, CHARGE=-4
  M_h_c: -2.0, CHARGE=1
  M_fadh2_c: 1.0, CHARGE=-2
  M_php2coa_c: 1.0, CHAR

In [ ]:
# Get sorted count dict
sorted_counts = sorted(count_dict.items(), key=lambda x: x[1], reverse=True)

In [ ]:
sorted_counts

[('M_h_c', 13),
 ('M_fad_c', 10),
 ('M_fadh2_c', 10),
 ('M_h2o_c', 4),
 ('M_atp_c', 2),
 ('M_adp_c', 2),
 ('M_pi_c', 2),
 ('M_3hnonacoa_c', 2),
 ('M_decoa_c', 1),
 ('M_dc2coa_c', 1),
 ('M_hptcoa_c', 1),
 ('M_hp2coa_c', 1),
 ('M_pdcacoa_c', 1),
 ('M_pdca2coa_c', 1),
 ('M_poctacoa_c', 1),
 ('M_pocta2coa_c', 1),
 ('M_phpcoa_c', 1),
 ('M_php2coa_c', 1),
 ('M_phxacoa_c', 1),
 ('M_phxa2coa_c', 1),
 ('M_pptcoa_c', 1),
 ('M_ppt2coa_c', 1),
 ('M_pbcoa_c', 1),
 ('M_pb2coa_c', 1),
 ('M_tdecoa_c', 1),
 ('M_tde2coa_c', 1),
 ('M_hdd4coa_c', 1),
 ('M_hdd4_2_coa_c', 1),
 ('M_dec4coa_c', 1),
 ('M_dec4_2_coa_c', 1),
 ('M_2dmmq8_c', 1),
 ('M_amet_c', 1),
 ('M_ahcys_c', 1),
 ('M_mql8_c', 1),
 ('M_fe3dhbzs3_p', 1),
 ('M_fe3dhbzs3_c', 1),
 ('M_nona2coa_c', 1),
 ('M_nad_c', 1),
 ('M_3ononacoa_c', 1),
 ('M_nadh_c', 1),
 ('M_2mecdp_c', 1),
 ('M_fdxrd_c', 1),
 ('M_fdxo_2_2_c', 1),
 ('M_h2mb4p_c', 1),
 ('M_dmlz_c', 1),
 ('M_4r5au_c', 1),
 ('M_ribflv_c', 1),
 ('M_salchs4fe_p', 1),
 ('M_salchs4fe_c', 1)]